# Macro Regime Sandbox

Play with macro feature combinations and build simple regime views outside the app.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from db.connection import get_engine
from stores.macro import MacroFeatureStore

In [ ]:
DATA_BACKEND = "local"
APP_ENV = "uat"
START_DATE = "2020-01-01"
FEATURES = [
    "UST_10Y_LEVEL",
    "UST_2S10S",
    "CPI_YOY",
    "BEI_5Y",
    "IG_OAS_LEVEL",
    "HY_OAS_LEVEL",
    "HY_MINUS_IG_OAS",
    "IG_OAS_Z20",
    "HY_OAS_Z20",
]

In [ ]:
engine = get_engine(data_backend=DATA_BACKEND, app_env=APP_ENV)
macro_feature_store = MacroFeatureStore(engine)
feature_matrix = macro_feature_store.get_feature_matrix(FEATURES, start_date=START_DATE)
feature_matrix.tail()

In [ ]:
latest = feature_matrix.dropna(how="all").iloc[-1]
latest

In [ ]:
regime_snapshot = {
    "duration_regime": "Bullish" if latest["UST_10Y_LEVEL"] < feature_matrix["UST_10Y_LEVEL"].tail(20).mean() else "Bearish",
    "curve_regime": "Steep" if latest["UST_2S10S"] > 0 else "Inverted",
    "inflation_regime": "Hot" if latest["CPI_YOY"] > 3 else "Cooling",
    "credit_regime": "Wide" if latest["HY_OAS_Z20"] > 1 else "Contained",
}
regime_snapshot

In [ ]:
plot_cols = ["UST_10Y_LEVEL", "UST_2S10S", "IG_OAS_LEVEL", "HY_OAS_LEVEL"]
fig, axes = plt.subplots(len(plot_cols), 1, figsize=(12, 10), sharex=True)
for ax, column in zip(axes, plot_cols):
    series = feature_matrix[column].dropna()
    if column.endswith("_OAS_LEVEL"):
        series = series * 100
    series.plot(ax=ax, title=column)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()